# GoEmotions-RoBERTa Training Pipeline

Linear pipeline: data engineering → baseline → RoBERTa fine-tuning → Captum XAI → export.

See [`docs/`](docs/) for detailed guides.

## Section 0: Environment Bootstrap (Kaggle / Local)

In [ ]:
import os
import sys
import subprocess

IS_KAGGLE = os.path.exists("/kaggle/input")
REPO_URL = "https://github.com/engrsakib/GoEmotions-RoBERTa-XAI-Fine-Tuned-Sentiment-Classifier-with-Token-Level-Attribution-Heatmaps.git"

if IS_KAGGLE:
    REPO_DIR = "/kaggle/working/repo"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    os.chdir(os.path.join(REPO_DIR, "notebooks"))
else:
    # Local: assume notebook is already in notebooks/
    if not os.path.basename(os.getcwd()) == "notebooks":
        nb_dir = os.path.join(os.getcwd(), "notebooks")
        if os.path.isdir(nb_dir):
            os.chdir(nb_dir)

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-train.txt"], check=True)
print(f"Working directory: {os.getcwd()}")
print(f"Kaggle environment: {IS_KAGGLE}")

## Section 1: Data Engineering

In [ ]:
from src.data.pipeline import run_data_pipeline, load_config

config = load_config()
data_result = run_data_pipeline(config)

train_df = data_result["train_df"]
val_df = data_result["val_df"]
test_df = data_result["test_df"]
stats = data_result["stats"]

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Leakage OK: {stats['leakage']['no_leakage']}")
print(f"Balance OK: {stats['balance']['within_tolerance']}")
print(f"Class counts: {stats['cleaning']['class_counts']}")

## Section 2: EDA Snapshot

In [ ]:
import matplotlib.pyplot as plt
from src.data.label_mapping import ID2LABEL

counts = train_df["encoded_label"].value_counts().sort_index()
labels = [ID2LABEL[i] for i in counts.index]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(labels, counts.values, color="steelblue")
ax.set_title("Training Set Class Distribution")
ax.set_ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print(f"Avg char length: {train_df['char_length'].mean():.1f}")
print(f"Avg token length (approx): {train_df['token_length_approx'].mean():.1f}")

## Section 3: Baseline — TF-IDF + Logistic Regression

In [ ]:
from src.training.baselines import train_baseline

baseline_result = train_baseline(
    train_df["text"], train_df["encoded_label"],
    test_df["text"], test_df["encoded_label"],
    max_features=config["baseline_max_features"],
    ngram_range=tuple(config["baseline_ngram_range"]),
)

baseline_macro_f1 = baseline_result["metrics"]["macro_f1"]
print(f"Baseline macro-F1: {baseline_macro_f1:.4f}")
print(baseline_result["classification_report"])

## Section 4: RoBERTa Fine-Tuning (Focal Loss + AdamW Cosine)

In [ ]:
import torch
from transformers import AutoTokenizer
from src.training.trainer_setup import build_trainer, prepare_hf_datasets

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

tokenizer = AutoTokenizer.from_pretrained(config["model_name"])
train_ds, val_ds, test_ds, train_labels = prepare_hf_datasets(
    train_df, val_df, test_df, tokenizer, max_length=config["max_length"]
)

trainer, tokenizer, model = build_trainer(
    config, train_ds, val_ds, train_labels=train_labels
)

print("Starting RoBERTa training...")
trainer.train()
print("Training complete.")

## Section 5: Test Evaluation

In [ ]:
from src.training.trainer_setup import evaluate_on_test

test_result = evaluate_on_test(trainer, test_ds)
roberta_macro_f1 = test_result["metrics"]["macro_f1"]

print(f"RoBERTa test macro-F1: {roberta_macro_f1:.4f}")
print(f"Baseline macro-F1:     {baseline_macro_f1:.4f}")
print(f"Improvement:           {roberta_macro_f1 - baseline_macro_f1:+.4f}")
print("\nClassification Report:")
print(test_result["classification_report"])
print("\nConfusion Matrix:")
print(test_result["confusion_matrix"])

## Section 6: XAI Validation — Captum Integrated Gradients

In [ ]:
from src.xai.captum_ig import explain_samples
from src.data.label_mapping import ID2LABEL

sample_texts = test_df["text"].sample(n=config["xai_sample_count"], random_state=42).tolist()
xai_results = explain_samples(
    model, tokenizer, sample_texts, device,
    max_length=config["max_length"],
    n_steps=config["ig_n_steps"],
)

for result in xai_results:
    label = ID2LABEL[result["target_class"]]
    top_tokens = sorted(zip(result["tokens"], result["heatmap"]), key=lambda x: abs(x[1]), reverse=True)[:5]
    print(f"\nText: {result['text'][:80]}...")
    print(f"Predicted: {label}")
    print(f"Top tokens: {top_tokens}")
    assert len(result["tokens"]) == len(result["heatmap"]), "Token/heatmap length mismatch"

## Section 7: Export Model

In [ ]:
from src.training.trainer_setup import export_model
from src.paths import EXPORTS_DIR

export_dir = export_model(trainer, tokenizer)
print(f"Model exported to: {export_dir}")
print(f"\nCopy to production:")
print(f"  cp -r {export_dir}/* ../packages/model/saved_emotion_model/")
print(f"\nOn Kaggle, download: {EXPORTS_DIR / 'saved_emotion_model'}")